In [1]:
# -*- coding: utf-8 -*-
import os
import sys
import gc
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==============================================================================
# 🎛️ PARAMETER UTAMA DIREKTORI (SINKRONISASI TOTAL)
# ==============================================================================
BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/rep_code"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")

# Mengunci Folder Acuan Statistik Laten 3C STEAD dari Zhi Geng
EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, STEAD norm7 mag3 L n61099, 30172538")

# File HDF5 Mahakarya 3C Sejati Indonesia hasil download barusan
PATH_DEMO_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo'
PATH_INPUT_HDF5 = os.path.join(PATH_DEMO_DIR, 'dataset_indonesia_sejati_3c_700.h5')
PATH_FINAL_REPORT = os.path.join(PATH_DEMO_DIR, 'mcu_quake_indonesia_true_3c_final_report.csv')

INPUT_SIZE = 700

# Menyuntikkan jalur repositori ke dalam system path Python interpreter
if PATH_DEMO_DIR not in sys.path: sys.path.append(PATH_DEMO_DIR)
if BASE_REP not in sys.path: sys.path.append(BASE_REP)

try:
    from Library import utils, dataset
    print("   ✅ HUBUNGAN PUSTAKA: Modul Library.utils & dataset berhasil terjalin!")
except ImportError as e:
    print(f"❌ GALAT SYSTEM PATH: Modul Library gagal termuat. Alasan: {str(e)}")
    sys.exit()

def jalankan_evaluasi_true_3c_indonesia():
    print("\n" + "="*90)
    print("🚀 STARTING: TRUE 3-COMPONENT (3C) INDEPENDENT BENCHMARK PIPELINE")
    print("="*90)
    
    if not os.path.exists(PATH_INPUT_HDF5):
        print(f"❌ GALAT: File database HDF5 3C tidak ditemukan di jalur: {PATH_INPUT_HDF5}")
        return

    # 1. Load Model Native (Kanal Tunggal Latent Extractor)
    print("⏳ Memuat arsitektur model biner MCU-Quake SavedModel...")
    try:
        embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
        print("   ✅ STATUS JARINGAN: Model pembaca biner sukses terpasang di RAM.")
    except Exception as e:
        print(f"❌ GALAT MEMUAT MODEL: {str(e)}")
        return
        
    # 2. Load File Matriks Acuan Statistik Laten 3C STEAD Global (n=61099)
    print("⏳ Memuat database statistik Typical Embeddings 3C STEAD...")
    try:
        embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
        embedding_N = dataset.load_embedding_data(EMB_DIR, "Embedding data, N.json")
        embedding_E = dataset.load_embedding_data(EMB_DIR, "Embedding data, E.json")
        
        # Kalkulasi kurva kepadatan probabilitas (KDE) multidimensional 3C
        embeddings_3C_PDFs = utils.embedding_PDFs_3D(embedding_Z, embedding_N, embedding_E)
        print("   ✅ STATUS DISTRIBUSI: Kurva KDE 3C STEAD berhasil dikunci di memori.")
    except Exception as e:
        print(f"❌ GALAT MEMUAT DATA EMBEDDING ACUAN: {str(e)}")
        return

    # 3. Membaca Database Tensor HDF5 Indonesia Sejati 3C
    print(f"\n⏳ Membuka berkas HDF5 dan menyedot tensor data...")
    with h5py.File(PATH_INPUT_HDF5, 'r') as hf:
        X_data = np.array(hf['X'])  # Shape: (Total_Sampel, 700, 3)
        Y_data = np.array(hf['Y'])  # Shape: (Total_Sampel,)
        names_data = np.array(hf['name'])
        
    total_sampel = len(X_data)
    print(f"   ✅ Sukses Memuat Tensor: Terdeteksi {total_sampel} total sampel array 3C siap uji.")

    # 4. Siklus Inferensi Instan di RAM
    y_true_list, y_pred_list = [], []
    report_rows = []
    
    print(f"\n⏳ Mengekstrak fitur ruang laten sekuensial & inferensi multi-dimensi KDE 3C...")
    # Karena pembacaan langsung dari matriks numpy, proses ini berjalan sangat cepat!
    for i in range(total_sampel):
        wave_3c = X_data[i]       # Matriks berukuran (700, 3)
        ground_truth = Y_data[i]   # 1 untuk Gempa, 0 untuk Noise
        ref_name = names_data[i].decode('utf-8')
        
        # Pisahkan komponen E, N, Z dari sumbu kanal (-1)
        # Sesuai aturan stack kita kemarin: axis-1 berisi [E, N, Z]
        tr_e = wave_3c[:, 0]
        tr_n = wave_3c[:, 1]
        tr_z = wave_3c[:, 2]
        
        try:
            # A. Ekstraksi Fitur Laten Menggunakan Fungsi Bawaan utils (Sekuensial 1C)
            _in_E = utils.latent_codes_1D(tr_e, embedding_model)
            _in_N = utils.latent_codes_1D(tr_n, embedding_model)
            _in_Z = utils.latent_codes_1D(tr_z, embedding_model)
            
            # B. Satukan ke Vektor Spasial 3C Fusion kaku Zhi Geng: np.array([E, N, Z])
            emb_3c = np.array([_in_E, _in_N, _in_Z]).reshape(1, -1)
            
            # C. Jalankan Inferensi Probabilitas Multi-Dimensi KDE 3C
            # p_3c mengembalikan indeks tebakan (0: Noise, 1: QB, 2: LE)
            p_3c, _, _ = utils.infer_3C_PDFs(emb_3c, embeddings_3C_PDFs, "Kernel")
            
            # D. Mapping Hasil ke Metrik Biner Berdasarkan Standar Replikasi STEAD (1 if p >= 1 else 0)
            pred_biner = 1 if p_3c >= 1 else 0
            
            y_true_list.append(ground_truth)
            y_pred_list.append(pred_biner)
            
            report_rows.append({
                'Station_Event_ID': ref_name,
                'Ground_Truth_Biner': ground_truth,
                'Predicted_Index_KDE': p_3c,
                'Final_Prediction_Biner': pred_biner
            })
        except Exception:
            continue

    # ==============================================================================
    # 📊 CALCULATE & DISPLAY FINAL SCIENTIFIC METRICS
    # ==============================================================================
    skor_akurasi = accuracy_score(y_true_list, y_pred_list)
    matriks_cm = confusion_matrix(y_true_list, y_pred_list)
    laporan_klasifikasi = classification_report(y_true_list, y_pred_list, target_names=['Noise (0)', 'Earthquake (1)'])
    
    print("\n📊" + "="*34 + " NATIVE RE-EVALUASI MATRIKS KEBINGUNGAN 3C " + "="*34)
    print(f"🎯 AKURASI GLOBAL SEJATI DATA INDONESIA 3C : {skor_akurasi * 100:.2f} %")
    print("-" * 112)
    print("📋 lAPORAN METRIK KLASIFIKASI RINCI DISERTASI:")
    print(laporan_klasifikasi)
    print("-" * 112)
    print("📊 MATRIKS KEBINGUNGAN NUMERIK (CONFUSION MATRIX):")
    print(matriks_cm)
    print("=" * 112)
    
    # Simpan hasil laporan berkas CSV final
    df_report = pd.DataFrame(report_rows)
    df_report.to_csv(PATH_FINAL_REPORT, index=False)
    print(f"📝 Tabel laporan evaluasi komparatif sukses dikunci di -> {PATH_FINAL_REPORT}\n")
    
    # 5. Visualisasi Grafik Publikasi Jurnal Internasional
    try:
        # Menghitung parameter internal untuk plot fungsi bawaan Zhi Geng jika dibutuhkan
        # Format matrix_3C buatan utils: [TN, FP, FN, TP] dalam bentuk skalar
        tn, fp, fn, tp = matriks_cm.ravel()
        matrix_formatted = np.array([[tn, fp], [fn, tp]])
        
        # Rekonstruksi kamus metrik tiruan agar kompatibel dengan plot_confusion bawaan
        mock_metrics = {
            'accuracy (avg.)': skor_akurasi,
            'f1-score (avg.)': (2 * tp) / (2 * tp + fp + fn)
        }
        
        fig_3C = utils.plot_confusion("MCU_5-20 Indonesia Real 3C Sejati KDE", ["NO", "LE"], matrix_formatted, mock_metrics)
        fig_3C.savefig(os.path.join(PATH_DEMO_DIR, "Indonesia_Data_True_3C_KDE_Final.jpg"), dpi=300)
        plt.show()
    except Exception as e:
        print(f"⚠️ Catatan Visualisasi: Plot otomatis dilewati, data teks di atas tetap 100% valid. Detail: {str(e)}")
        
    del df_report, embedding_model, X_data, Y_data
    gc.collect()

if __name__ == "__main__":
    jalankan_evaluasi_true_3c_indonesia()

   ✅ HUBUNGAN PUSTAKA: Modul Library.utils & dataset berhasil terjalin!

🚀 STARTING: TRUE 3-COMPONENT (3C) INDEPENDENT BENCHMARK PIPELINE
❌ GALAT: File database HDF5 3C tidak ditemukan di jalur: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo/dataset_indonesia_sejati_3c_700.h5


In [2]:
# -*- coding: utf-8 -*-
import os
import sys
import gc
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==============================================================================
# 🎛️ PARAMETER UTAMA DIREKTORI (SINKRONISASI TOTAL)
# ==============================================================================
BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/rep_code"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")

# Mengunci Folder Acuan Statistik Laten 3C STEAD dari Zhi Geng
EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, STEAD norm7 mag3 L n61099, 30172538")

# File HDF5 Mahakarya Baru yang Bebas Bias Hasil Download Barusan
PATH_DEMO_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo'
PATH_INPUT_HDF5 = os.path.join(PATH_DEMO_DIR, 'dataset_indonesia_sejati_3c_700.h5')
PATH_FINAL_REPORT = os.path.join(PATH_DEMO_DIR, 'mcu_quake_indonesia_true_3c_recovered_report.csv')

INPUT_SIZE = 700

# Menyuntikkan jalur repositori ke dalam system path Python interpreter
if PATH_DEMO_DIR not in sys.path: sys.path.append(PATH_DEMO_DIR)
if BASE_REP not in sys.path: sys.path.append(BASE_REP)

try:
    from Library import utils, dataset
    print("   ✅ HUBUNGAN PUSTAKA: Modul Library.utils & dataset berhasil terjalin!")
except ImportError as e:
    print(f"❌ GALAT SYSTEM PATH: Modul Library gagal termuat. Alasan: {str(e)}")
    sys.exit()

def jalankan_evaluasi_true_3c_recovered():
    print("\n" + "="*90)
    print("🚀 STARTING: FINAL RECOVERED TRUE 3C BENCHMARK PIPELINE")
    print("="*90)
    
    if not os.path.exists(PATH_INPUT_HDF5):
        print(f"❌ GALAT: File database HDF5 tidak ditemukan di jalur: {PATH_INPUT_HDF5}")
        return

    # 1. Load Model Native (Kanal Tunggal Latent Extractor)
    print("⏳ Memuat arsitektur model biner MCU-Quake SavedModel...")
    try:
        embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
        print("   ✅ STATUS JARINGAN: Model pembaca biner sukses terpasang di RAM.")
    except Exception as e:
        print(f"❌ GALAT MEMUAT MODEL: {str(e)}")
        return
        
    # 2. Load File Matriks Acuan Statistik Laten 3C STEAD Global (n=61099)
    print("⏳ Memuat database statistik Typical Embeddings 3C STEAD...")
    try:
        embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
        embedding_N = dataset.load_embedding_data(EMB_DIR, "Embedding data, N.json")
        embedding_E = dataset.load_embedding_data(EMB_DIR, "Embedding data, E.json")
        
        # Kalkulasi kurva kepadatan probabilitas (KDE) multidimensional 3C
        embeddings_3C_PDFs = utils.embedding_PDFs_3D(embedding_Z, embedding_N, embedding_E)
        print("   ✅ STATUS DISTRIBUSI: Kurva KDE 3C STEAD berhasil dikunci di memori.")
    except Exception as e:
        print(f"❌ GALAT MEMUAT DATA EMBEDDING ACUAN: {str(e)}")
        return

    # 3. Membaca Database Tensor HDF5 Baru Indonesia Sejati 3C
    print(f"\n⏳ Membuka berkas HDF5 baru dan menyedot tensor data...")
    with h5py.File(PATH_INPUT_HDF5, 'r') as hf:
        X_data = np.array(hf['X'])  # Shape: (Total_Sampel, 700, 3)
        Y_data = np.array(hf['Y'])  # Shape: (Total_Sampel,)
        names_data = np.array(hf['name'])
        
    total_sampel = len(X_data)
    print(f"   ✅ Sukses Memuat Tensor: Terdeteksi {total_sampel} total sampel array 3C bebas bias.")

    # 4. Siklus Inferensi Instan di RAM
    y_true_list, y_pred_list = [], []
    report_rows = []
    
    print(f"\n⏳ Mengekstrak fitur ruang laten sekuensial & inferensi multi-dimensi KDE 3C...")
    for i in range(total_sampel):
        wave_3c = X_data[i]       # Matriks berukuran (700, 3)
        ground_truth = Y_data[i]   # 1 untuk Gempa, 0 untuk Noise
        ref_name = names_data[i].decode('utf-8')
        
        # Pisahkan komponen E, N, Z dari sumbu kanal akhir
        # Berdasarkan susunan stack skrip download: axis-1 berisi [E, N, Z]
        tr_e = wave_3c[:, 0]
        tr_n = wave_3c[:, 1]
        tr_z = wave_3c[:, 2]
        
        try:
            # A. Ekstraksi Fitur Laten Menggunakan Fungsi Bawaan utils (Sekuensial 1C)
            _in_E = utils.latent_codes_1D(tr_e, embedding_model)
            _in_N = utils.latent_codes_1D(tr_n, embedding_model)
            _in_Z = utils.latent_codes_1D(tr_z, embedding_model)
            
            # B. Satukan ke Vektor Spasial 3C Fusion kaku Zhi Geng: np.array([E, N, Z])
            emb_3c = np.array([_in_E, _in_N, _in_Z]).reshape(1, -1)
            
            # C. Jalankan Inferensi Probabilitas Multi-Dimensi KDE 3C
            p_3c, _, _ = utils.infer_3C_PDFs(emb_3c, embeddings_3C_PDFs, "Kernel")
            
            # D. Mapping Hasil ke Metrik Biner Berdasarkan Standar Replikasi STEAD (1 if p >= 1 else 0)
            pred_biner = 1 if p_3c >= 1 else 0
            
            y_true_list.append(ground_truth)
            y_pred_list.append(pred_biner)
            
            report_rows.append({
                'Station_Event_ID': ref_name,
                'Ground_Truth_Biner': ground_truth,
                'Predicted_Index_KDE': p_3c,
                'Final_Prediction_Biner': pred_biner
            })
        except Exception:
            continue

    # ==============================================================================
    # 📊 CALCULATE & DISPLAY FINAL SCIENTIFIC METRICS
    # ==============================================================================
    skor_akurasi = accuracy_score(y_true_list, y_pred_list)
    matriks_cm = confusion_matrix(y_true_list, y_pred_list)
    laporan_klasifikasi = classification_report(y_true_list, y_pred_list, target_names=['Noise (0)', 'Earthquake (1)'])
    
    print("\n📊" + "="*34 + " FINAL VERIFIED 3C CONFUSION MATRIX " + "="*34)
    print(f"🎯 AKURASI RE-EVALUASI AKHIR DATA INDONESIA 3C : {skor_akurasi * 100:.2f} %")
    print("-" * 112)
    print("📋 LAPORAN METRIK KLASIFIKASI RINCI DISERTASI:")
    print(laporan_klasifikasi)
    print("-" * 112)
    print("📊 MATRIKS KEBINGUNGAN NUMERIK (CONFUSION MATRIX):")
    print(matriks_cm)
    print("=" * 112)
    
    # Simpan hasil laporan berkas CSV final
    df_report = pd.DataFrame(report_rows)
    df_report.to_csv(PATH_FINAL_REPORT, index=False)
    print(f"📝 Tabel laporan evaluasi komparatif sukses dikunci di -> {PATH_FINAL_REPORT}\n")
    
    # 5. Visualisasi Grafik Publikasi Jurnal Internasional
    try:
        tn, fp, fn, tp = matriks_cm.ravel()
        matrix_formatted = np.array([[tn, fp], [fn, tp]])
        
        mock_metrics = {
            'accuracy (avg.)': skor_akurasi,
            'f1-score (avg.)': (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
        }
        
        fig_3C = utils.plot_confusion("MCU_5-20 Indonesia Recovered 3C KDE", ["NO", "LE"], matrix_formatted, mock_metrics)
        fig_3C.savefig(os.path.join(PATH_DEMO_DIR, "Indonesia_Data_Recovered_3C_KDE.jpg"), dpi=300)
        plt.show()
    except Exception as e:
        print(f"⚠️ Catatan Visualisasi: Plot otomatis dilewati, data teks tetap 100% aman. Detail: {str(e)}")
        
    del df_report, embedding_model, X_data, Y_data
    gc.collect()

if __name__ == "__main__":
    # 🔥 PERBAIKAN MUTLAK: Pemanggilan fungsi disamakan 100% dengan deklarasi di atas
    jalankan_evaluasi_true_3c_recovered()

   ✅ HUBUNGAN PUSTAKA: Modul Library.utils & dataset berhasil terjalin!

🚀 STARTING: FINAL RECOVERED TRUE 3C BENCHMARK PIPELINE
❌ GALAT: File database HDF5 tidak ditemukan di jalur: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo/dataset_indonesia_sejati_3c_700.h5
